# Predictive Maintenance: RUL Motor Induksi (XGBoost)



## 1. Setup environment (Colab / VS Code)

In [ ]:
import sys, subprocess, importlib, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

def pastikan(paket, impor=None):
    """Install paket hanya jika belum ada (aman untuk Colab & lokal)."""
    try:
        importlib.import_module(impor or paket)
    except ImportError:
        print("Menginstal", paket)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paket])

pastikan("xgboost"); pastikan("scikit-learn", "sklearn"); pastikan("pandas")
pastikan("matplotlib"); pastikan("joblib")

if IN_COLAB:
    ROOT = Path("/content/mkb-kasus3-rul")
else:
    ROOT = Path.cwd()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent
(ROOT / "data").mkdir(parents=True, exist_ok=True)
(ROOT / "artifacts").mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print("Colab:", IN_COLAB, "| ROOT:", ROOT)

## 2. Muat dataset



In [ ]:
DATA_PATH = ROOT / "data" / "dataset_motor_induksi_rul.csv"

if not DATA_PATH.exists() and IN_COLAB:
    from google.colab import files
    print("Unggah dataset_motor_induksi_rul.csv")
    up = files.upload()
    nama = next(iter(up))
    DATA_PATH.write_bytes(up[nama])

assert DATA_PATH.exists(), f"Dataset tidak ditemukan di {DATA_PATH}"

import pandas as pd, numpy as np
df = pd.read_csv(DATA_PATH)
FITUR_INPUT = ["arus_ampere", "getaran_mm_s", "suhu_body_c", "tegangan_volt"]
TARGET = "sisa_umur_jam"
kurang = [c for c in FITUR_INPUT + [TARGET] if c not in df.columns]
assert not kurang, f"Kolom hilang: {kurang}"

print("Ukuran data:", df.shape)
if len(df) < 50:
    print(f"⚠️ PERINGATAN: hanya {len(df)} baris. Metrik tidak bermakna; pakai dataset penuh.")
df.head()

### (Opsional) Data sintetis untuk uji alur


In [ ]:
PAKAI_DATA_SINTETIS = False

if PAKAI_DATA_SINTETIS:
    rng = np.random.default_rng(42)
    n = 1000
    kerusakan = rng.uniform(0, 1, n)
    df = pd.DataFrame({
        "arus_ampere":   10 + 12 * kerusakan + rng.normal(0, 0.8, n),
        "getaran_mm_s":  1 + 10 * kerusakan**1.5 + rng.normal(0, 0.4, n),
        "suhu_body_c":   40 + 45 * kerusakan + rng.normal(0, 3, n),
        "tegangan_volt": 220 + rng.normal(0, 2, n),
    })
    df["sisa_umur_jam"] = (2500 * (1 - kerusakan) ** 1.3 + rng.normal(0, 40, n)).clip(1).round(2)

    df.loc[rng.choice(n, 30, replace=False), "suhu_body_c"] = np.nan
    df.loc[rng.choice(n, 10, replace=False), "getaran_mm_s"] = -1
    print("Memakai DATA SINTETIS:", df.shape)

## 3. Eksplorasi data (EDA)

In [ ]:
print("Missing value per kolom:"); print(df.isna().sum(), "\n")
print("Nilai negatif (sensor error):")
print({c: int((df[c] < 0).sum()) for c in ["arus_ampere", "getaran_mm_s"]}, "\n")
df.describe().T

In [ ]:
import matplotlib.pyplot as plt

if len(df) >= 10:
    fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
    for a, c in zip(ax, FITUR_INPUT):
        a.scatter(df[c], df[TARGET], s=8, alpha=.5)
        a.set_xlabel(c); a.set_ylabel(TARGET)
    plt.tight_layout(); plt.show()
    print(df.corr(numeric_only=True)[TARGET].round(3))
else:
    print("Data terlalu sedikit untuk plot yang berarti.")

## 4. Pembersihan & pipeline



In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

def bersihkan_data(df):
    """Nilai negatif pada arus/getaran dianggap sensor error -> NaN (diimputasi median)."""
    df_bersih = df.copy()
    df_bersih.loc[df_bersih["getaran_mm_s"] < 0, "getaran_mm_s"] = np.nan
    df_bersih.loc[df_bersih["arus_ampere"] < 0, "arus_ampere"] = np.nan
    return df_bersih

def buat_pipeline():
    return Pipeline([
        ("cleaning", FunctionTransformer(bersihkan_data)),
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)),
    ])

## 5. Latih & evaluasi

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X, y = df[FITUR_INPUT], df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = buat_pipeline().fit(X_tr, y_tr)
base = Pipeline([("imputer", SimpleImputer(strategy="median")),
                 ("m", DummyRegressor(strategy="mean"))]).fit(X_tr, y_tr)

pred = pipe.predict(X_te).clip(min=0)
metrik = {
    "mae_model":    float(mean_absolute_error(y_te, pred)),
    "rmse_model":   float(np.sqrt(mean_squared_error(y_te, pred))),
    "r2_model":     float(r2_score(y_te, pred)) if len(y_te) > 1 else None,
    "mae_baseline": float(mean_absolute_error(y_te, base.predict(X_te))),
    "n_train": len(X_tr), "n_test": len(X_te),
}
for k, v in metrik.items():
    print(f"{k:14s}: {v}")

In [ ]:
if len(y_te) >= 10:
    plt.figure(figsize=(5, 5))
    plt.scatter(y_te, pred, s=10, alpha=.6)
    lim = [0, max(y_te.max(), pred.max())]
    plt.plot(lim, lim, "r--", label="ideal")
    plt.xlabel("RUL aktual (jam)"); plt.ylabel("RUL prediksi (jam)"); plt.legend(); plt.show()

    imp = pd.Series(pipe.named_steps["model"].feature_importances_, index=FITUR_INPUT).sort_values()
    imp.plot.barh(title="Feature importance"); plt.show()

## 6. Simpan model portabel + metadata


In [ ]:
import json, joblib, sklearn, xgboost

SIMPAN_KE_ARTIFACTS = False
OUT_DIR = ROOT / "artifacts" if SIMPAN_KE_ARTIFACTS else ROOT / "artifacts" / "retrained"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pipe.named_steps["model"].get_booster().save_model(str(OUT_DIR / "xgb_rul.json"))
joblib.dump(pipe.named_steps["imputer"], OUT_DIR / "imputer_rul.joblib")

meta = {
    "versi_model": "1.0.0",
    "algoritma": "XGBRegressor (n_estimators=300, lr=0.05, max_depth=5) dalam Pipeline cleaning->imputer(median)->model",
    "format_model": "booster XGBoost (JSON) + imputer (joblib)",
    "fitur_input": FITUR_INPUT,
    "target": TARGET,
    "metrik_test": metrik,
    "data_sintetis": PAKAI_DATA_SINTETIS,
    "library": {"scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__,
                "pandas": pd.__version__, "numpy": np.__version__},
}
(OUT_DIR / "metadata.json").write_text(json.dumps(meta, indent=2))
for f in ("xgb_rul.json", "imputer_rul.joblib", "metadata.json"):
    print("Tersimpan:", OUT_DIR / f, f"({(OUT_DIR / f).stat().st_size/1024:.0f} KB)")

## 7. Uji prediksi (muat ulang model dari disk)
Status: `PRIMA` > 500 jam · `NORMAL` > 100 · `PERHATIAN` > 24 · `KRITIS` ≤ 24.


In [ ]:
import xgboost as xgb

def status_dari_rul(rul):
    return "PRIMA" if rul > 500 else "NORMAL" if rul > 100 else "PERHATIAN" if rul > 24 else "KRITIS"

def muat_portabel(folder):
    booster = xgb.Booster()
    booster.load_model(str(folder / "xgb_rul.json"))
    imputer = joblib.load(folder / "imputer_rul.joblib")
    def prediksi(df_in):
        x = imputer.transform(bersihkan_data(df_in[FITUR_INPUT]))
        return booster.predict(xgb.DMatrix(np.asarray(x, dtype="float32")))
    return prediksi

prediksi = muat_portabel(OUT_DIR)

selisih = float(np.abs(prediksi(X) - pipe.predict(X)).max())
print(f"Selisih maks. model disk vs memori: {selisih:.6f}")
assert selisih < 1e-3, "Model dari disk tidak sama dengan pipeline di memori!"

kasus = {
    "normal":  dict(arus_ampere=12.5, getaran_mm_s=2.1,  suhu_body_c=45, tegangan_volt=220),
    "ekstrem": dict(arus_ampere=20,   getaran_mm_s=12,   suhu_body_c=95, tegangan_volt=200),
    "sensor error (arus -5)": dict(arus_ampere=-5, getaran_mm_s=2.1, suhu_body_c=45, tegangan_volt=220),
}
for nama, d in kasus.items():
    rul = max(0.0, float(prediksi(pd.DataFrame([d]))[0]))
    print(f"{nama:24s} -> {rul:10.2f} jam  [{status_dari_rul(rul)}]")

## 8. Unduh file



In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    zip_path = shutil.make_archive("/content/model_rul_portabel", "zip", OUT_DIR)
    files.download(zip_path)
else:
    print("Model ada di:", OUT_DIR)